# NLP: текстовые пайплайны

Очистка текста, TF-IDF бейзлайн, Transformers fine-tuning (HuggingFace).

## Импорты

In [ ]:
import re
import random
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Очистка текста

In [ ]:
def clean_text(text):
    text = str(text).lower()
    # заклинания которые вряд ли нужны, но могут помочь добрать последние баллы
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# train['clean_text'] = train['text'].apply(clean_text)
# test['clean_text'] = test['text'].apply(clean_text)

## Быстрый бейзлайн: TF-IDF + LogisticRegression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

tfidf_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        sublinear_tf=True,
    )),
    ('clf', LogisticRegression(C=1.0, max_iter=1000)),
])

# Кросс-валидация
# scores = cross_val_score(tfidf_pipe, train['clean_text'], train['target'], cv=5, scoring='f1_macro')
# print(f'TF-IDF baseline CV F1: {scores.mean():.4f} +/- {scores.std():.4f}')

# Обучение + предсказание
# tfidf_pipe.fit(train['clean_text'], train['target'])
# test_preds = tfidf_pipe.predict(test['clean_text'])

---
## Transformers Fine-Tuning (HuggingFace)

Для более серьёзного качества. Требует GPU.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler

MODEL_NAME = 'bert-base-uncased'  # или: microsoft/deberta-v3-base, cointegrated/rubert-tiny2
MAX_LEN = 128
BATCH_SIZE = 32
NUM_EPOCHS = 5
LR = 2e-5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

### Dataset для текстов

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
        }
        if 'token_type_ids' in enc:
            item['token_type_ids'] = enc['token_type_ids'].squeeze()
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = TextDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_ds = TextDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_ds = TextDataset(test_texts, None, tokenizer, MAX_LEN)

# num_workers - количество потоков для загрузки данных, 0 - один поток, если поставить слишком много, можно упасть в ошибку

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

### Модель + оптимизатор

In [ ]:
NUM_CLASSES = 3

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler = GradScaler()

### Train / Eval

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        token_type_ids = batch.get('token_type_ids')
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)

        # forward pass 
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        loss = criterion(outputs.logits, labels)

        # backward pass
        loss.backward()
        
        # Клиппинг градиентов, скорее всего не нужно
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        # Шаг оптимизатора и планировщика
        optimizer.step()
        scheduler.step() # вряд ли нужно, скорее если вы добираете последние баллы
        optimizer.zero_grad()

        total_loss += loss.item()
        all_preds.extend(outputs.logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), f1_score(all_labels, all_preds, average='macro')



@torch.no_grad()
def evaluate_transformer(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        token_type_ids = batch.get('token_type_ids')
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)

        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
            loss = criterion(outputs.logits, labels)

        total_loss += loss.item()
        all_preds.extend(outputs.logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), f1_score(all_labels, all_preds, average='macro')

In [ ]:
best_f1 = 0
for epoch in range(NUM_EPOCHS):
    train_loss, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, device)
    val_loss, val_f1 = evaluate_transformer(model, val_loader, criterion, device)
    print(f'Epoch {epoch+1} | train_loss={train_loss:.4f} train_f1={train_f1:.4f} | val_loss={val_loss:.4f} val_f1={val_f1:.4f}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'best_nlp_model.pt')

print(f'\nBest val F1: {best_f1:.4f}')
model.load_state_dict(torch.load('best_nlp_model.pt', map_location=device))

### Предсказание + Submission

In [ ]:
@torch.no_grad()
def predict_transformer(model, loader, device):
    model.eval()
    all_probs = []
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch.get('token_type_ids')
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
    return np.vstack(all_probs)

test_probs = predict_transformer(model, test_loader, device)
test_preds = test_probs.argmax(axis=1)

submission = pd.DataFrame({'id': test_ids, 'target': test_preds})
submission.to_csv('submission.csv', index=False)
submission.head()